# Model 3 — 05 Production Artifact & Metadata

## Purpose
Create the exact production artifact structure requested.

This notebook is the release step. It retrains final pipelines on the deterministic training cache, writes the deployable `network_threat_model.pkl`, and generates environment, schema, label, configuration, metric and version metadata.

In [1]:

from pathlib import Path
import sys, json, os, warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "EVIDENT" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "behaviour"
MODEL_DIR = PROJECT_ROOT / "models" / "behavior"
ENV_DIR = MODEL_DIR / "environments"
META_DIR = MODEL_DIR / "metadata"
PROD_DIR = MODEL_DIR / "production"

for d in (ENV_DIR, META_DIR, PROD_DIR):
    d.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset folder not found: {DATA_DIR}")

import pandas as pd
import numpy as np

def clean_col(c):
    return " ".join(str(c).strip().replace("\ufeff","").split())

def norm_label(x):
    return "UNKNOWN" if pd.isna(x) else str(x).strip().upper().replace(" ","_")

def family(x):
    x = norm_label(x)
    if x == "BENIGN": return "BENIGN"
    if "PATATOR" in x or "BRUTE" in x: return "BRUTE_FORCE"
    if "PORTSCAN" in x or "PORT_SCAN" in x: return "PORT_SCAN"
    if "DDOS" in x or x.startswith("DOS"): return "DOS_DDOS"
    if "WEB_ATTACK" in x or "SQL_INJECTION" in x or "XSS" in x: return "WEB_ATTACK"
    if "BOT" in x: return "BOTNET"
    if "INFILTRATION" in x: return "INFILTRATION"
    if "HEARTBLEED" in x: return "HEARTBLEED"
    return "OTHER_ATTACK"

files = sorted(DATA_DIR.glob("*.csv"))
print("EVIDENT root:", PROJECT_ROOT)
print("CIC-IDS2017 files:", len(files))
for p in files:
    print(" -", p.name)
assert files, "No CIC-IDS2017 CSV files found."


EVIDENT root: c:\Users\bimal\OneDrive\Desktop\EVIDENT
CIC-IDS2017 files: 8
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv


In [2]:

import pandas as pd, json, joblib, platform, sys, sklearn, numpy
from datetime import datetime, timezone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier

cache=PROJECT_ROOT/"data"/"processed"/"behavior"/"cicids2017_model3_training.parquet"
schema=json.loads((META_DIR/"feature_schema.json").read_text())
df=pd.read_parquet(cache)
features=schema["feature_columns"]

def model(n):
    return Pipeline([
        ("imputer",SimpleImputer(strategy="median")),
        ("scaler",StandardScaler()),
        ("classifier",HistGradientBoostingClassifier(
            max_iter=n,learning_rate=.08,max_leaf_nodes=31,
            l2_regularization=1.0,random_state=42))
    ])

X=df[features]; yb=df["is_attack"]; yf=df["attack_family"]
binary=model(250).fit(X,yb)
mask=yf!="BENIGN"
family_model=model(300).fit(X[mask],yf[mask])

artifact={
 "artifact_type":"EVIDENT_MODEL3_NETWORK_THREAT_DETECTOR",
 "artifact_version":"1.0.0",
 "binary_model":binary,
 "family_model":family_model,
 "feature_columns":features,
 "family_labels":sorted(yf[mask].unique().tolist())
}
joblib.dump(artifact,PROD_DIR/"network_threat_model.pkl",compress=3)

label_mapping={
 "binary":{"0":"BENIGN","1":"ATTACK"},
 "attack_family":{
  "BRUTE_FORCE":"Brute-force / password guessing",
  "DOS_DDOS":"Denial of Service / Distributed Denial of Service",
  "PORT_SCAN":"Port scanning / reconnaissance",
  "WEB_ATTACK":"Web application attack",
  "BOTNET":"Botnet traffic",
  "INFILTRATION":"Network infiltration",
  "HEARTBLEED":"Heartbleed-style attack",
  "OTHER_ATTACK":"Other attack family"
 }
}
(META_DIR/"label_mapping.json").write_text(json.dumps(label_mapping,indent=2),encoding="utf-8")

training_config={
 "model_name":"EVIDENT Network Threat Detector",
 "dataset":"CIC-IDS2017",
 "random_state":42,
 "preprocessing":["numeric coercion","inf->NaN","median imputation","StandardScaler"],
 "binary_algorithm":"HistGradientBoostingClassifier",
 "family_algorithm":"HistGradientBoostingClassifier",
 "feature_count":len(features)
}
(META_DIR/"training_config.json").write_text(json.dumps(training_config,indent=2),encoding="utf-8")

env={
 "created_at_utc":datetime.now(timezone.utc).isoformat(),
 "python_version":sys.version,
 "platform":platform.platform(),
 "python_executable":sys.executable,
 "pandas_version":pd.__version__,
 "numpy_version":numpy.__version__,
 "scikit_learn_version":sklearn.__version__,
 "joblib_version":joblib.__version__,
 "dataset":"CIC-IDS2017"
}
(ENV_DIR/"training_environment.json").write_text(json.dumps(env,indent=2),encoding="utf-8")

req="""pandas>=2.0
numpy>=1.24
scikit-learn>=1.3
joblib>=1.3
jupyter>=1.0
matplotlib>=3.7
seaborn>=0.13
"""
(ENV_DIR/"requirements_model3.txt").write_text(req,encoding="utf-8")

metrics=json.loads((META_DIR/"evaluation_metrics.json").read_text()) if (META_DIR/"evaluation_metrics.json").exists() else {}
metadata={
 "model_name":"EVIDENT Network Threat Detector",
 "model_version":"1.0.0",
 "model_type":"supervised_network_intrusion_detection",
 "dataset":"CIC-IDS2017",
 "feature_count":len(features),
 "targets":["is_attack","attack_family"],
 "production_artifact":"models/behavior/production/network_threat_model.pkl",
 "metadata_generated_utc":datetime.now(timezone.utc).isoformat(),
 "evaluation_summary":{
   "binary_f1":metrics.get("binary",{}).get("f1"),
   "binary_recall":metrics.get("binary",{}).get("recall"),
   "attack_family_accuracy":metrics.get("attack_family",{}).get("accuracy")
 }
}
(META_DIR/"model_metadata.json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")

version={
 "model_name":"EVIDENT Network Threat Detector",
 "version":"1.0.0",
 "status":"production",
 "artifact":"network_threat_model.pkl",
 "dataset":"CIC-IDS2017",
 "created_at_utc":datetime.now(timezone.utc).isoformat()
}
(PROD_DIR/"model_version.json").write_text(json.dumps(version,indent=2),encoding="utf-8")

required=[
 ENV_DIR/"training_environment.json",ENV_DIR/"requirements_model3.txt",
 META_DIR/"model_metadata.json",META_DIR/"feature_schema.json",
 META_DIR/"label_mapping.json",META_DIR/"training_config.json",
 META_DIR/"evaluation_metrics.json",PROD_DIR/"network_threat_model.pkl",
 PROD_DIR/"model_version.json"
]
missing=[str(p) for p in required if not p.exists()]
assert not missing, missing
print("Production release complete.")
for p in required:
    print(" -",p.relative_to(PROJECT_ROOT))


Production release complete.
 - models\behavior\environments\training_environment.json
 - models\behavior\environments\requirements_model3.txt
 - models\behavior\metadata\model_metadata.json
 - models\behavior\metadata\feature_schema.json
 - models\behavior\metadata\label_mapping.json
 - models\behavior\metadata\training_config.json
 - models\behavior\metadata\evaluation_metrics.json
 - models\behavior\production\network_threat_model.pkl
 - models\behavior\production\model_version.json
